In [ ]:
!pip install pandas numpy scikit-learn openpyxl

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

# To debug, let's first check if the directory exists and list its contents
# This will help verify the correct path and filename.
import os

file_path = '/content/drive/MyDrive/DrSeba/Dr.Seba_500_Organized_Final.xlsx'
directory_path = os.path.dirname(file_path)

if not os.path.exists(directory_path):
    print(f"Error: Directory '{directory_path}' does not exist. Please check your Google Drive path.")
elif not os.path.isfile(file_path):
    print(f"Error: File '{file_path}' not found. Listing contents of '{directory_path}':")
    try:
        print(os.listdir(directory_path))
    except Exception as e:
        print(f"Could not list directory contents: {e}")
else:
    try:
        df = pd.read_excel(file_path)
        print("Dataset shape:", df.shape)
        print(df.head(15))
    except Exception as e:
        print(f"Error reading Excel file: {e}")

Dataset shape: (500, 21)
    thana_rank doctor_id                      doctor_name district      thana  \
0            1   DOC0032            Dr. Md. Ahsanul Hoque  Barisal  Bakerganj   
1            2   DOC0358              Dr. Md. Kamruzzaman  Barisal  Bakerganj   
2            3   DOC0020          Dr. Md. Motlabur Rahman  Barisal  Bakerganj   
3            4   DOC0198              Dr. Md. Nahid Hasan  Barisal  Bakerganj   
4            5   DOC0124    Prof. Dr. Md. Haider Ali Khan  Barisal  Bakerganj   
5            6   DOC0232                 Dr. Ariful Haque  Barisal  Bakerganj   
6            7   DOC0058  Prof. Dr. Abul Kashem Khandaker  Barisal  Bakerganj   
7            8   DOC0417           Prof. Dr. F M Siddiqui  Barisal  Bakerganj   
8            9   DOC0320    Prof. Dr. Md. Haider Ali Khan  Barisal  Bakerganj   
9           10   DOC0151       Dr. A.Z.M. Mahfuzur Rahman  Barisal  Bakerganj   
10          11   DOC0220     Dr. Hafiz Ahmed Nazmul Hakim  Barisal  Bakerganj   
11 

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Handle missing values
df['rating_avg'] = df['rating_avg'].fillna(df['rating_avg'].mean())
df['experience_years'] = df['experience_years'].fillna(df['experience_years'].median())
df['consultation_fees'] = df['consultation_fees'].fillna(df['consultation_fees'].median()) # Corrected column name
df['rating_count'] = df['rating_count'].fillna(0) # Corrected column name, assuming total_reviews maps to rating_count

# Convert binary features
df['online_consultation'] = df['online_consultation'].map({'Yes':1, 'No':0})
df['emergency_service'] = df['emergency_service'].map({'Yes':1, 'No':0})

# Drop rows with critical missing values
df = df.dropna(subset=['specialization_group'])

In [ ]:
# Normalize rating and experience
scaler = MinMaxScaler()
df[['rating_norm', 'experience_norm']] = scaler.fit_transform(
    df[['rating_avg', 'experience_years']]
)

# Log transform reviews
df['reviews_log'] = np.log1p(df['rating_count'])

# Final target score (IMPORTANT)
df['target_score'] = (
    0.5 * df['rating_norm'] +
    0.3 * df['experience_norm'] +
    0.2 * df['reviews_log']
)

df[['rating_norm', 'experience_norm', 'reviews_log', 'target_score']].head()

,rating_norm,experience_norm,reviews_log,target_score
0,0.973154,0.476190,5.673323,1.764099
1,0.946309,0.857143,5.442418,1.818781
2,0.852349,0.047619,4.584967,1.357454
3,0.691275,0.666667,5.676754,1.680988
4,0.657718,0.238095,4.644391,1.329166


In [ ]:
le_spec = LabelEncoder()
le_hosp = LabelEncoder()

df['specialization_group'] = le_spec.fit_transform(df['specialization_group'].astype(str))
df['hospital_type'] = le_hosp.fit_transform(df['hospital_type'].astype(str))

In [ ]:
features = [
    'experience_years',
    'consultation_fees',
    'rating_avg',
    'rating_count',
    'specialization_group',
    'hospital_type',
    'online_consultation',
    'emergency_service',
    'hospital_name',
    'district',
    'thana'
]

X = df[features]
y = df['target_score']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Categorical columns that need encoding and are still in string format
categorical_cols_for_encoding_in_X = ['hospital_name', 'district', 'thana']

# Store encoders globally
global feature_encoders
feature_encoders = {}

# Create and apply LabelEncoders to X_train and X_test
for col in categorical_cols_for_encoding_in_X:
    le = LabelEncoder()

    # Fit the encoder on the training data and transform X_train
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    feature_encoders[col] = le # Store the fitted encoder

    # Transform X_test using the *same* encoder fitted on X_train
    # Handle unseen categories in X_test by mapping them to -1
    label_mapping = {label: idx for idx, label in enumerate(le.classes_)}
    X_test[col] = X_test[col].astype(str).map(label_mapping).fillna(-1).astype(int)

# Now, all features in X_train and X_test are numerical.
model.fit(X_train, y_train)

GradientBoostingRegressor(learning_rate=0.05, max_depth=4, n_estimators=200,
                          random_state=42)

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Performance:")
print("RMSE:", round(rmse, 4))
print("R2 Score:", round(r2, 4))

Model Performance:
RMSE: 0.0227
R2 Score: 0.9879


In [ ]:
import pandas as pd

importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance)

                 Feature  Importance
2             rating_avg    0.505210
3           rating_count    0.297049
0       experience_years    0.195957
5          hospital_type    0.000448
4   specialization_group    0.000379
10                 thana    0.000282
6    online_consultation    0.000231
9               district    0.000186
8          hospital_name    0.000143
1      consultation_fees    0.000097
7      emergency_service    0.000017


In [ ]:
from sklearn.preprocessing import LabelEncoder

def recommend_doctors(user_input, df, model, top_n=5):

    temp_df = df.copy()

    # Filter by specialization
    try:
        # le_spec is a global variable from cell M-mtfHx6Ue-4
        spec_encoded = le_spec.transform([user_input['specialization']])[0]
        temp_df = temp_df[temp_df['specialization_group'] == spec_encoded]
    except ValueError: # Catch ValueError if specialization not found
        return "Specialization not found"

    # Filter by budget
    temp_df = temp_df[temp_df['consultation_fees'] <= user_input['max_fee']]

    # Filter online if needed
    if user_input['online'] == 1:
        temp_df = temp_df[temp_df['online_consultation'] == 1]

    # Filter by district if provided
    if 'district' in user_input and user_input['district'] is not None:
        temp_df = temp_df[temp_df['district'] == user_input['district']]

    # Filter by thana if provided
    if 'thana' in user_input and user_input['thana'] is not None:
        temp_df = temp_df[temp_df['thana'] == user_input['thana']]

    if len(temp_df) == 0:
        return "No doctors found with given criteria"

    # Prepare X_temp for prediction
    X_temp = temp_df[features].copy() # Make a copy to avoid SettingWithCopyWarning

    # Encode categorical features in X_temp that the model expects as numbers
    categorical_cols_for_encoding_in_X = ['hospital_name', 'district', 'thana']
    for col in categorical_cols_for_encoding_in_X:
        # Use the global encoder stored during training
        le = feature_encoders[col]

        # Transform X_temp; handle unseen values if any by mapping them to -1
        label_mapping = {label: idx for idx, label in enumerate(le.classes_)}
        X_temp[col] = X_temp[col].astype(str).map(label_mapping).fillna(-1).astype(int)

    # Predict scores
    temp_df['predicted_score'] = model.predict(X_temp)

    # Sort and return
    result = temp_df.sort_values(by='predicted_score', ascending=False)

    return result[['doctor_name', 'rating_avg', 'experience_years',
                   'consultation_fees', 'predicted_score','hospital_name','full_address']].head(top_n)

In [ ]:
user_input = {
    'district': 'Barisal',
    'thana': 'Bakerganj',
    'specialization': 'Surgeon',  # must match dataset
    'max_fee': 2000,
    'online': 0,
    'emergency': 0

}

recommend_doctors(user_input, df, model, top_n=5)

,doctor_name,rating_avg,experience_years,consultation_fees,predicted_score,hospital_name,full_address
3,Dr. Md. Nahid Hasan,4.53,17,2000,1.677530,Lancet Diagnostic Center,"837 Hospital Lane, Bakerganj, Barisal"
17,Dr. Md. Fazlul Haque Qasem,3.64,22,600,1.446785,Lancet Diagnostic Center,"424/A Bakerganj Road, Barisal"
13,Dr. AMM Mukarrabin,3.91,14,1800,1.392002,Community Based Medical College Hospital,"767/A Bakerganj Road, Barisal"
21,Dr. Md. Akter,3.52,17,500,1.131229,Gazi Medical College Hospital,"777 Hospital Lane, Bakerganj, Barisal"


In [ ]:
!pip install catboost
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import AdaBoostRegressor

# XGBoost
print("\n--- XGBoost Model Performance ---")
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)
print("RMSE (XGBoost):", round(rmse_xgb, 4))
print("R2 Score (XGBoost):", round(r2_xgb, 4))

# CatBoost
print("\n--- CatBoost Model Performance ---")
cat_model = CatBoostRegressor(iterations=200, learning_rate=0.05, depth=4, random_state=42, verbose=False)
cat_model.fit(X_train, y_train)
y_pred_cat = cat_model.predict(X_test)
rmse_cat = np.sqrt(mean_squared_error(y_test, y_pred_cat))
r2_cat = r2_score(y_test, y_pred_cat)
print("RMSE (CatBoost):", round(rmse_cat, 4))
print("R2 Score (CatBoost):", round(r2_cat, 4))

# AdaBoost
print("\n--- AdaBoost Model Performance ---")
ada_model = AdaBoostRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
ada_model.fit(X_train, y_train)
y_pred_ada = ada_model.predict(X_test)
rmse_ada = np.sqrt(mean_squared_error(y_test, y_pred_ada))
r2_ada = r2_score(y_test, y_pred_ada)
print("RMSE (AdaBoost):", round(rmse_ada, 4))
print("R2 Score (AdaBoost):", round(r2_ada, 4))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.3 MB/s eta 0:00:00

--- XGBoost Model Performance ---
RMSE (XGBoost): 0.0235
R2 Score (XGBoost): 0.987

--- CatBoost Model Performance ---
RMSE (CatBoost): 0.0126
R2 Score (CatBoost): 0.9963

--- AdaBoost Model Performance ---
RMSE (AdaBoost): 0.0793
R2 Score (AdaBoost): 0.852


In [ ]:
print("\n--- Model Accuracy Comparison ---")
print("Gradient Boosting Regressor:")
print(f"  RMSE: {rmse:.4f}")
print(f"  R2 Score: {r2:.4f}")

print("\nXGBoost Regressor:")
print(f"  RMSE: {rmse_xgb:.4f}")
print(f"  R2 Score: {r2_xgb:.4f}")

print("\nCatBoost Regressor:")
print(f"  RMSE: {rmse_cat:.4f}")
print(f"  R2 Score: {r2_cat:.4f}")

print("\nAdaBoost Regressor:")
print(f"  RMSE: {rmse_ada:.4f}")
print(f"  R2 Score: {r2_ada:.4f}")


--- Model Accuracy Comparison ---
Gradient Boosting Regressor:
  RMSE: 0.0227
  R2 Score: 0.9879

XGBoost Regressor:
  RMSE: 0.0235
  R2 Score: 0.9870

CatBoost Regressor:
  RMSE: 0.0126
  R2 Score: 0.9963

AdaBoost Regressor:
  RMSE: 0.0793
  R2 Score: 0.8520


In [ ]:
user_input = {
    'district': 'Barisal',
    'thana': 'Bakerganj',
    'specialization': 'Surgeon',  # must match dataset
    'max_fee': 1500,
    'online': 0,
    'emergency': 0
}

# Get recommendations using the CatBoost model
recommend_doctors(user_input, df, cat_model, top_n=5)

,doctor_name,rating_avg,experience_years,consultation_fees,predicted_score,hospital_name,full_address
17,Dr. Md. Fazlul Haque Qasem,3.64,22,600,1.457347,Lancet Diagnostic Center,"424/A Bakerganj Road, Barisal"
21,Dr. Md. Akter,3.52,17,500,1.144687,Gazi Medical College Hospital,"777 Hospital Lane, Bakerganj, Barisal"


In [ ]:
import pickle

model_package = {
    "model": cat_model,   # BEST MODEL
    "le_spec": le_spec,
    "le_hosp": le_hosp,
    "feature_encoders": feature_encoders,
    "features": features
}

with open("doctor_ai_full_package.pkl", "wb") as f:
    pickle.dump(model_package, f)

print("Full model package saved!")

Full model package saved!


In [ ]:
from google.colab import files
files.download("doctor_ai_full_package.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_path = "/content/drive/MyDrive/DrSeba/doctor_ai_full_package.pkl"

with open(drive_path, "wb") as f:
    pickle.dump(model_package, f)

print("Saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive!
